In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Project configuration
CATALOG = "food_delivery_catalog"

SOURCE_SCHEMA = "source"
BRONZE_SCHEMA = "bronze"

VOLUME_PATH = "/Volumes/food_delivery_catalog/source/food_delivery_files/"

# Source files
INITIAL_FILE = VOLUME_PATH + "food_delivery_initial_100.csv"
INCREMENTAL_FILE = VOLUME_PATH + "food_delivery_incremental_batch2_50.csv"

# Batch identifiers
BATCH_1 = "BATCH_001"

# Target tables
RAW_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.orders_raw"
VALID_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.orders_valid"
REJECTED_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.orders_rejected"
AUDIT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.reconciliation_audit"

print("Project configuration completed.")


Project configuration completed.


In [0]:

# Read initial dataset
df_initial = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(INITIAL_FILE)
    .withColumn("batch_id", F.lit(BATCH_1))
)

print("Initial records:", df_initial.count())



Initial records: 100


In [0]:

df_bronze = (
    df_initial
    .withColumn(
        "source_file",
        F.lit("food_delivery_initial_100.csv")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

display(df_bronze.limit(10))


order_id,order_date,customer_id,restaurant_name,city,food_category,quantity,item_price,delivery_fee,discount,order_status,batch_id,source_file,ingestion_timestamp
ORD0001,2026-10-05,C001,Restaurant 2,Mumbai,CHINESE,2.0,375.0,90.0,20.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0002,2026-10-06,C002,Restaurant 3,Kochi,CONTINENTAL,3.0,550.0,130.0,40.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0003,2026-10-07,C003,Restaurant 4,Delhi,INDIAN,4.0,725.0,170.0,60.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0004,2026-10-08,C004,Restaurant 5,Coimbatore,CHINESE,5.0,900.0,210.0,0.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0005,2026-10-09,C005,Restaurant 6,Bengaluru,CONTINENTAL,1.0,1075.0,250.0,20.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0006,2026-10-10,C006,Restaurant 7,Chennai,INDIAN,2.0,1250.0,290.0,40.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0007,2026-10-11,C007,Restaurant 8,Hyderabad,CHINESE,3.0,1425.0,50.0,60.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0008,2026-10-12,C008,Restaurant 1,Pune,CONTINENTAL,4.0,200.0,90.0,0.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0009,2026-10-13,C009,Restaurant 2,Mumbai,INDIAN,5.0,375.0,130.0,20.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z
ORD0010,2026-10-14,C010,Restaurant 3,Kochi,CHINESE,1.0,550.0,170.0,40.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:37.082Z


In [0]:

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_TABLE)
)

print("Raw Bronze count:", spark.table(RAW_TABLE).count())


Raw Bronze count: 100


In [0]:

# ============================================================
# BRONZE LAYER - DATA VALIDATION, DUPLICATE HANDLING
# AND RECONCILIATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Read raw Bronze data
df = spark.table(RAW_TABLE)

# 2. Convert fields temporarily for validation
df_check = (
    df
    .withColumn("_order_date", F.expr("try_cast(order_date AS DATE)"))
    .withColumn("_quantity", F.expr("try_cast(quantity AS DOUBLE)"))
    .withColumn("_item_price", F.expr("try_cast(item_price AS DOUBLE)"))
    .withColumn("_delivery_fee", F.expr("try_cast(delivery_fee AS DOUBLE)"))
    .withColumn("_discount", F.expr("try_cast(discount AS DOUBLE)"))
)

# 3. Accepted order statuses
valid_statuses = [
    "DELIVERED",
    "CANCELLED",
    "PLACED",
    "PREPARING",
    "OUT_FOR_DELIVERY"
]

# 4. Build rejection reasons
rejection_reasons = [
    F.when(
        F.col("order_id").isNull() |
        (F.trim("order_id") == ""),
        "MISSING_ORDER_ID"
    ),
    F.when(
        F.col("customer_id").isNull() |
        (F.trim("customer_id") == ""),
        "MISSING_CUSTOMER_ID"
    ),
    F.when(
        F.col("restaurant_name").isNull() |
        (F.trim("restaurant_name") == ""),
        "MISSING_RESTAURANT_NAME"
    ),
    F.when(
        F.col("city").isNull() |
        (F.trim("city") == ""),
        "MISSING_CITY"
    ),
    F.when(
        F.col("food_category").isNull() |
        (F.trim("food_category") == ""),
        "MISSING_FOOD_CATEGORY"
    ),
    F.when(
        F.col("_order_date").isNull(),
        "INVALID_ORDER_DATE"
    ),
    F.when(
        F.col("_quantity").isNull() |
        (F.col("_quantity") <= 0) |
        (F.col("_quantity") != F.floor("_quantity")),
        "INVALID_QUANTITY"
    ),
    F.when(
        F.col("_item_price").isNull() |
        (F.col("_item_price") <= 0),
        "INVALID_ITEM_PRICE"
    ),
    F.when(
        F.col("_delivery_fee").isNull() |
        (F.col("_delivery_fee") < 0),
        "INVALID_DELIVERY_FEE"
    ),
    F.when(
        F.col("_discount").isNull() |
        (F.col("_discount") < 0) |
        (
            F.col("_quantity").isNotNull() &
            F.col("_item_price").isNotNull() &
            F.col("_delivery_fee").isNotNull() &
            (
                F.col("_discount") >
                (
                    F.col("_quantity") *
                    F.col("_item_price") +
                    F.col("_delivery_fee")
                )
            )
        ),
        "INVALID_DISCOUNT"
    ),
    F.when(
        F.col("order_status").isNull() |
        (~F.upper(F.trim("order_status")).isin(valid_statuses)),
        "INVALID_ORDER_STATUS"
    )
]

# 5. Generate original validation reasons
df_validated = df_check.withColumn(
    "rejection_reason",
    F.concat_ws("; ", *rejection_reasons)
)


In [0]:


# 6. Create a deterministic hash for each source record
hash_columns = [
    F.col(c)
    for c in df_validated.columns
    if c != "rejection_reason"
]

df_validated = df_validated.withColumn(
    "_record_hash",
    F.sha2(
        F.to_json(F.struct(*hash_columns)),
        256
    )
)

# 7. Rank duplicates deterministically
duplicate_window = Window.partitionBy("order_id").orderBy(
    F.when(F.col("rejection_reason") == "", 0)
     .otherwise(1).asc(),
    F.col("ingestion_timestamp").asc(),
    F.col("_record_hash").asc()
)

df_check = df_validated.withColumn(
    "duplicate_rank",
    F.row_number().over(duplicate_window)
)

# 8. Append duplicate rejection reason
df_check = df_check.withColumn(
    "rejection_reason",
    F.when(
        F.col("order_id").isNull() |
        (F.trim("order_id") == ""),
        F.col("rejection_reason")
    ).when(
        F.col("duplicate_rank") > 1,
        F.when(
            F.col("rejection_reason") == "",
            F.lit("DUPLICATE_ORDER_ID")
        ).otherwise(
            F.concat(
                F.col("rejection_reason"),
                F.lit("; DUPLICATE_ORDER_ID")
            )
        )
    ).otherwise(F.col("rejection_reason"))
).drop("duplicate_rank", "_record_hash")


# 8. Split valid and rejected records
df_valid = (
    df_check
    .filter(F.col("rejection_reason") == "")
    .drop(
        "_order_date",
        "_quantity",
        "_item_price",
        "_delivery_fee",
        "_discount",
        "rejection_reason"
    )
)

df_rejected = (
    df_check
    .filter(F.col("rejection_reason") != "")
    .drop(
        "_order_date",
        "_quantity",
        "_item_price",
        "_delivery_fee",
        "_discount"
    )
)

# 9. Write valid records
(
    df_valid.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VALID_TABLE)
)

# 10. Write rejected records
(
    df_rejected.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJECTED_TABLE)
)

# 11. Reconciliation
source_count = df.count()
valid_count = df_valid.count()
rejected_count = df_rejected.count()

reconciled_count = valid_count + rejected_count

reconciliation_status = (
    "PASS" if source_count == reconciled_count else "FAIL"
)

print("SOURCE COUNT:", source_count)
print("VALID COUNT:", valid_count)
print("REJECTED COUNT:", rejected_count)
print("RECONCILED COUNT:", reconciled_count)
print("RECONCILIATION STATUS:", reconciliation_status)


SOURCE COUNT: 100
VALID COUNT: 83
REJECTED COUNT: 17
RECONCILED COUNT: 100
RECONCILIATION STATUS: PASS


In [0]:
%sql
SELECT *
FROM food_delivery_catalog.bronze.orders_rejected
ORDER BY order_id;

order_id,order_date,customer_id,restaurant_name,city,food_category,quantity,item_price,delivery_fee,discount,order_status,batch_id,source_file,ingestion_timestamp,rejection_reason
null,2026-10-22,C033,Restaurant 5,Kochi,MEXICAN,3.0,275.0,75.0,30.0,OUT_FOR_DELIVERY,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,MISSING_ORDER_ID
null,2026-10-12,C036,Restaurant 5,Coimbatore,INDIAN,2.0,900.0,90.0,0.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,MISSING_ORDER_ID
ORD0037,2026-10-13,C037,null,Bengaluru,CHINESE,3.0,1075.0,130.0,20.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,MISSING_RESTAURANT_NAME
ORD0038,2026-10-14,C038,Restaurant 7,Chennai,CONTINENTAL,-1.0,1250.0,170.0,40.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,INVALID_QUANTITY; INVALID_DISCOUNT
ORD0039,2026-10-15,C039,Restaurant 8,Hyderabad,INDIAN,5.0,1425.0,210.0,60.0,INVALID_VALUE,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,INVALID_ORDER_STATUS; DUPLICATE_ORDER_ID
ORD0041,2026-10-01,null,Restaurant 1,Chennai,CONTINENTAL,1.0,180.0,35.0,0.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,MISSING_CUSTOMER_ID
ORD0045,2026-10-05,C045,Restaurant 5,,MEXICAN,5.0,375.0,75.0,30.0,PREPARING,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,MISSING_CITY; DUPLICATE_ORDER_ID
ORD0049,2026-10-09,C049,Restaurant 1,Chennai,INDIAN,-2.0,725.0,55.0,20.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,INVALID_QUANTITY; INVALID_DISCOUNT
ORD0053,2026-10-13,C053,Restaurant 5,Kochi,CONTINENTAL,3.0,0.0,35.0,0.0,OUT_FOR_DELIVERY,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,INVALID_ITEM_PRICE
ORD0057,2026-10-17,C057,Restaurant 1,Chennai,MEXICAN,2.0,525.0,-15.0,30.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T06:28:45.651Z,INVALID_DELIVERY_FEE


In [0]:
from pyspark.sql import functions as F

BATCH2_PATH = "/Volumes/food_delivery_catalog/source/food_delivery_files/food_delivery_incremental_batch2_50.csv"

BRONZE_RAW_TABLE = "food_delivery_catalog.bronze.orders_raw"
BATCH_ID = "BATCH_002"

# Check whether this batch has already been ingested
existing_batch_count = (
    spark.table(BRONZE_RAW_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
    .count()
)

if existing_batch_count > 0:
    print(f"{BATCH_ID} already exists in Bronze.")
    print("Skipping ingestion to prevent duplicate records.")

else:
    # Read Batch 2
    df_batch2 = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(BATCH2_PATH)
    )

    # Add metadata
    df_batch2 = (
        df_batch2
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("source_file", F.lit("food_delivery_incremental_batch2_50.csv"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    # Match Bronze schema
    df_batch2 = df_batch2.select(
        "order_id",
        "order_date",
        "customer_id",
        "restaurant_name",
        "city",
        "food_category",
        "quantity",
        "item_price",
        "delivery_fee",
        "discount",
        "order_status",
        "batch_id",
        "source_file",
        "ingestion_timestamp"
    )

    # Append only once
    df_batch2.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(BRONZE_RAW_TABLE)

    print("Batch 2 successfully ingested!")

# Final verification
display(
    spark.table(BRONZE_RAW_TABLE)
    .groupBy("batch_id")
    .count()
)

BATCH_002 already exists in Bronze.
Skipping ingestion to prevent duplicate records.


batch_id,count
BATCH_001,100
BATCH_002,50


In [0]:
%sql
SELECT
    batch_id,
    COUNT(*) AS total_records
FROM food_delivery_catalog.bronze.orders_raw
GROUP BY batch_id
ORDER BY batch_id;

batch_id,total_records
BATCH_001,100
BATCH_002,50


In [0]:
%sql
SELECT
    'VALID' AS record_type,
    COUNT(*) AS total_records
FROM food_delivery_catalog.dbt_work.batch2_valid

UNION ALL

SELECT
    'REJECTED' AS record_type,
    COUNT(*) AS total_records
FROM food_delivery_catalog.dbt_work.batch2_rejected;

record_type,total_records
VALID,46
REJECTED,4


In [0]:
%sql
SELECT
    order_id,
    customer_id,
    rejection_reason,
    batch_id
FROM food_delivery_catalog.dbt_work.batch2_rejected
ORDER BY order_id;

order_id,customer_id,rejection_reason,batch_id
ORD0112,C044,INVALID_QUANTITY; INVALID_DISCOUNT,BATCH_002
ORD0118,C050,INVALID_ORDER_STATUS,BATCH_002
ORD0126,C058,MISSING_CITY,BATCH_002
ORD0132,C004,INVALID_DISCOUNT,BATCH_002


In [0]:
%sql
SELECT
    rejection_reason,
    COUNT(*) AS rejected_count
FROM food_delivery_catalog.dbt_work.batch2_rejected
GROUP BY rejection_reason
ORDER BY rejected_count DESC;

rejection_reason,rejected_count
MISSING_CITY,1
INVALID_ORDER_STATUS,1
INVALID_QUANTITY; INVALID_DISCOUNT,1
INVALID_DISCOUNT,1


In [0]:
%sql
SELECT
    CASE
        WHEN s.order_id IS NOT NULL THEN 'UPDATE'
        ELSE 'INSERT'
    END AS operation_type,
    COUNT(*) AS record_count
FROM food_delivery_catalog.dbt_work.batch2_valid b
LEFT JOIN food_delivery_catalog.silver.orders_cleaned s
    ON b.order_id = s.order_id
GROUP BY
    CASE
        WHEN s.order_id IS NOT NULL THEN 'UPDATE'
        ELSE 'INSERT'
    END;

operation_type,record_count
UPDATE,15
INSERT,31


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders
FROM food_delivery_catalog.dbt_work_silver.orders_cleaned_dbt;

total_orders,unique_orders
114,114


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders
FROM food_delivery_catalog.dbt_work_silver.orders_cleaned_dbt;

total_orders,unique_orders
114,114


In [0]:
%sql
DESCRIBE TABLE food_delivery_catalog.gold.dim_restaurant;



col_name,data_type,comment
restaurant_sk,int,null
restaurant_id,string,null
restaurant_name,string,null
restaurant_city,string,null
primary_cuisine,string,null
service_status,string,null
valid_from,date,null
valid_to,date,null
is_current,boolean,null
# Clustering Information,,


In [0]:
%sql
DESCRIBE TABLE food_delivery_catalog.gold.dim_city;

col_name,data_type,comment
city_sk,int,null
city_name,string,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql
DESCRIBE TABLE food_delivery_catalog.gold.dim_food_category;

col_name,data_type,comment
food_category_sk,int,null
category_name,string,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql
DESCRIBE TABLE food_delivery_catalog.gold.dim_date;

col_name,data_type,comment
order_date,date,null
date_sk,int,null
year,int,null
month,int,null
month_name,string,null
quarter,int,null
day,int,null
day_name,string,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql
SELECT
    COUNT(*) AS total_fact_rows,
    COUNT(DISTINCT order_id) AS unique_orders,

    SUM(CASE WHEN restaurant_sk IS NULL THEN 1 ELSE 0 END)
        AS unmatched_restaurants,

    SUM(CASE WHEN city_sk IS NULL THEN 1 ELSE 0 END)
        AS unmatched_cities,

    SUM(CASE WHEN food_category_sk IS NULL THEN 1 ELSE 0 END)
        AS unmatched_categories,

    SUM(CASE WHEN date_sk IS NULL THEN 1 ELSE 0 END)
        AS unmatched_dates,

    ROUND(SUM(order_amount), 2) AS total_revenue

FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt;

total_fact_rows,unique_orders,unmatched_restaurants,unmatched_cities,unmatched_categories,unmatched_dates,total_revenue
114,114,0,0,0,23,214801.00


In [0]:
%sql
SELECT
    r.restaurant_name,
    r.service_status,
    COUNT(f.order_id) AS total_orders,
    ROUND(SUM(f.order_amount), 2) AS total_revenue
FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt f
JOIN food_delivery_catalog.gold.dim_restaurant r
    ON f.restaurant_sk = r.restaurant_sk
GROUP BY
    r.restaurant_name,
    r.service_status
ORDER BY total_revenue DESC;

restaurant_name,service_status,total_orders,total_revenue
Restaurant 8,ACTIVE,15,35362.00
Restaurant 7,ACTIVE,14,34052.00
Restaurant 4,ACTIVE,18,31691.00
Restaurant 2,ACTIVE,17,25871.00
Restaurant 3,ACTIVE,16,24812.00
Restaurant 5,ACTIVE,8,21406.00
Restaurant 6,ACTIVE,8,19895.00
Restaurant 6,INACTIVE,8,12908.00
Restaurant 1,ACTIVE,10,8804.00


In [0]:
%sql
SELECT
    s.order_date,
    COUNT(*) AS affected_orders
FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt f
JOIN food_delivery_catalog.dbt_work_silver.orders_cleaned_dbt s
    ON f.order_id = s.order_id
WHERE f.date_sk IS NULL
GROUP BY s.order_date
ORDER BY s.order_date;

order_date,affected_orders
2026-11-01,2
2026-11-02,2
2026-11-03,2
2026-11-05,2
2026-11-06,2
2026-11-07,2
2026-11-08,1
2026-11-09,1
2026-11-11,1
2026-11-12,1


In [0]:
%sql
SELECT
    MIN(order_date) AS earliest_date,
    MAX(order_date) AS latest_date,
    COUNT(*) AS total_dates
FROM food_delivery_catalog.gold.dim_date;

earliest_date,latest_date,total_dates
2026-10-01,2026-10-31,31


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_revenue
FROM food_delivery_catalog.dbt_work_silver.orders_cleaned_dbt;

total_orders,total_revenue
114,214801.00


In [0]:
%sql
SELECT
    COUNT(*) AS total_fact_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN date_sk IS NULL THEN 1 ELSE 0 END) AS unmatched_dates,
    SUM(CASE WHEN restaurant_sk IS NULL THEN 1 ELSE 0 END) AS unmatched_restaurants,
    ROUND(SUM(order_amount), 2) AS total_revenue
FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt;

total_fact_rows,unique_orders,unmatched_dates,unmatched_restaurants,total_revenue
114,114,0,0,214801.00


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders,
    ROUND(SUM(order_amount), 2) AS total_revenue
FROM food_delivery_catalog.dbt_work_silver.orders_cleaned_merge_demo;

total_orders,unique_orders,total_revenue
83,83,159225.00


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders,
    ROUND(SUM(order_amount), 2) AS total_revenue,
    COUNT_IF(batch_id = 'BATCH_002') AS batch2_orders
FROM food_delivery_catalog.dbt_work_silver.orders_cleaned_merge_demo;

total_orders,unique_orders,total_revenue,batch2_orders
114,114,214801.00,46


In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/food_delivery_catalog/source/food_delivery_files/restaurants_initial.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

restaurant_id,restaurant_name,restaurant_city,primary_cuisine,service_status,_rescued_data
RST001,Restaurant 1,Bengaluru,INDIAN,ACTIVE,null
RST002,Restaurant 2,Mumbai,CHINESE,ACTIVE,null
RST003,Restaurant 3,Kochi,CONTINENTAL,ACTIVE,null
RST004,Restaurant 4,Delhi,INDIAN,ACTIVE,null
RST005,Restaurant 5,Coimbatore,CHINESE,ACTIVE,null
RST006,Restaurant 6,Bengaluru,CONTINENTAL,ACTIVE,null
RST007,Restaurant 7,Chennai,INDIAN,ACTIVE,null
RST008,Restaurant 8,Hyderabad,CHINESE,ACTIVE,null


In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/food_delivery_catalog/source/food_delivery_files/restaurants_changes.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

restaurant_id,restaurant_name,restaurant_city,primary_cuisine,service_status,effective_date,_rescued_data
RST003,Restaurant 3,Kochi,CONTINENTAL,ACTIVE,2026-10-20,null
RST006,Restaurant 6,Bengaluru,CONTINENTAL,INACTIVE,2026-10-22,null
